# Final Supervised Fine-Tuning (SFT) Experiment

This notebook runs the final SFT condition for the SFT-vs-DPO comparison. It uses the shared data-preparation and evaluation utilities in `src/` so that SFT and DPO use the same 5,000 valid HH-RLHF training pairs and the same 1,000 valid HH-RLHF test pairs.


## 1. Install dependencies

Run this cell in Google Colab or a fresh environment. It intentionally does not install PyTorch, CUDA, or NumPy.


In [ ]:
%pip install -q \
    transformers==4.53.3 \
    datasets==3.6.0 \
    accelerate==1.8.1 \
    trl==0.19.1 \
    huggingface-hub==0.36.2 \
    fsspec==2025.3.0


## 2. Import shared project code


In [ ]:
from pathlib import Path
import os
import sys

# Keep framework selection explicit in Colab/local notebooks.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Shared src available:", (PROJECT_ROOT / "src").exists())


## 3. Environment and seed


In [ ]:
import torch
import transformers
import datasets
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Seed:", SEED)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Shared HH-RLHF split preparation

Parsing and validity filtering happen on the official HH-RLHF train/test splits before the final 5,000/1,000 samples are selected. The sampled `source_index` values are deterministic under seed 42.


In [ ]:
from transformers import AutoTokenizer
from src.data_preparation import (
    MODEL_NAME,
    DATASET_NAME,
    TRAIN_SIZE,
    TEST_SIZE,
    MAX_LENGTH,
    MAX_PROMPT_LENGTH,
    prepare_hh_rlhf_splits,
)

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)
print("Train pairs:", TRAIN_SIZE)
print("Test pairs:", TEST_SIZE)
print("Max sequence length:", MAX_LENGTH)
print("Max prompt length:", MAX_PROMPT_LENGTH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

default_tokenizer_length = getattr(tokenizer, "model_max_length", None)
print("Tokenizer pad token:", tokenizer.pad_token)
print("Tokenizer model_max_length:", default_tokenizer_length)

train_pairs, test_pairs, split_info = prepare_hh_rlhf_splits(tokenizer)

assert len(train_pairs) == TRAIN_SIZE
assert len(test_pairs) == TEST_SIZE
assert len(set(train_pairs["source_index"])) == TRAIN_SIZE
assert len(set(test_pairs["source_index"])) == TEST_SIZE

print("Split information:")
for key, value in split_info.as_dict().items():
    print(f"  {key}: {value}")

print("First five train source indices:", train_pairs["source_index"][:5])
print("First five test source indices: ", test_pairs["source_index"][:5])


## 5. Load the base model


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

param_count = sum(p.numel() for p in model.parameters())
print("Loaded model:", MODEL_NAME)
print("Parameter count:", f"{param_count:,}")
print("Dtype:", next(model.parameters()).dtype)


## 6. Evaluate the base model on the shared test set


In [ ]:
from src.evaluation import evaluate_preference_pairs, print_preference_summary

base_rows, base_stats = evaluate_preference_pairs(
    model=model,
    tokenizer=tokenizer,
    pairs=test_pairs,
    output_csv=PROJECT_ROOT / "results" / "base_preference_eval.csv",
    model_label="Base model",
    max_length=MAX_LENGTH,
)
print_preference_summary("BASE MODEL", base_stats)
print("Saved per-example CSV:", PROJECT_ROOT / "results" / "base_preference_eval.csv")


## 7. Build the SFT training dataset from the shared training pairs

SFT uses only the chosen response from each of the 5,000 shared training preference pairs.


In [ ]:
from src.data_preparation import to_sft_text_dataset, tokenize_sft_example

sft_text = to_sft_text_dataset(train_pairs)
sft_tokenized = sft_text.map(
    lambda ex: tokenize_sft_example(ex, tokenizer, max_length=MAX_LENGTH),
    remove_columns=sft_text.column_names,
    desc="Tokenizing SFT examples",
)

examples_without_targets = sum(all(label == -100 for label in labels) for labels in sft_tokenized["labels"])
max_sequence_length = max(len(ids) for ids in sft_tokenized["input_ids"])

assert len(sft_tokenized) == TRAIN_SIZE
assert examples_without_targets == 0
assert max_sequence_length <= MAX_LENGTH

print("SFT optimization examples:", len(sft_tokenized))
print("Examples without supervised response tokens:", examples_without_targets)
print("Maximum tokenized sequence length:", max_sequence_length)


## 8. SFT training configuration


In [ ]:
from transformers import DataCollatorForSeq2Seq, TrainingArguments
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

training_args = TrainingArguments(
    output_dir="./sft_model",
    num_train_epochs=1,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    weight_decay=0.01,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    logging_steps=10,
    logging_strategy="steps",
    save_strategy="epoch",
    report_to="none",
    fp16=False,
    bf16=False,
    seed=SEED,
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt",
)

train_loader = DataLoader(
    sft_tokenized,
    batch_size=training_args.per_device_train_batch_size,
    shuffle=True,
    collate_fn=data_collator,
)

optimizer = AdamW(
    model.parameters(),
    lr=training_args.learning_rate,
    weight_decay=training_args.weight_decay,
)

num_training_steps = int(
    np.ceil(len(train_loader) / training_args.gradient_accumulation_steps)
    * training_args.num_train_epochs
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=training_args.warmup_steps,
    num_training_steps=num_training_steps,
)

print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Per-device train batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Effective batch size:", training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
print("Optimizer: AdamW")
print("Weight decay:", training_args.weight_decay)
print("Scheduler:", training_args.lr_scheduler_type)
print("Warmup steps:", training_args.warmup_steps)
print("Planned optimizer steps:", num_training_steps)
print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)


## 9. Train SFT


In [ ]:
from tqdm.auto import tqdm
import time

model.train()
train_losses = []
optimizer.zero_grad()
start_time = time.time()

for step, batch in enumerate(tqdm(train_loader, desc="SFT training"), start=1):
    batch = {k: v.to(device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss / training_args.gradient_accumulation_steps
    loss.backward()
    train_losses.append(loss.item() * training_args.gradient_accumulation_steps)

    should_step = (
        step % training_args.gradient_accumulation_steps == 0
        or step == len(train_loader)
    )
    if should_step:
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

elapsed = time.time() - start_time
print("SFT training complete")
print("Batches:", len(train_loader))
print("Optimizer steps:", scheduler.last_epoch)
print("Mean training loss:", float(np.mean(train_losses)))
print("Elapsed seconds:", round(elapsed, 2))


## 10. Save the SFT model


In [ ]:
SAVE_DIR = PROJECT_ROOT / "models" / "smollm2_sft_final"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved SFT model:", SAVE_DIR)


## 11. Evaluate SFT on the shared test set

The metric is the same shared mean response-token log-probability margin used for the base model and DPO.


In [ ]:
sft_rows, sft_stats = evaluate_preference_pairs(
    model=model,
    tokenizer=tokenizer,
    pairs=test_pairs,
    output_csv=PROJECT_ROOT / "results" / "sft_preference_eval.csv",
    model_label="SFT model",
    max_length=MAX_LENGTH,
)
print_preference_summary("SFT MODEL", sft_stats)
print("Saved per-example CSV:", PROJECT_ROOT / "results" / "sft_preference_eval.csv")
